# miniGPT — Complete Version: up to the 10-gram, with back-off

**Ways of Thinking with Data and Computation · Habib University**

---

Session 3 stopped at the trigram and left two questions open:

> *If we went on to 4-grams, 5-grams, 6-grams — what would happen?*
>
> *When the trigram is stuck, the bigram might still know an answer. How could
> you make the model fall back on it?*

This notebook answers both. It builds **every** model from the unigram up to
the **10-gram**, and then generates text with **back-off** (the "regression"
step): if the 10-gram has never seen the last 9 words, drop the leftmost word
and ask the 9-gram about the last 8, and so on down to the unigram, which can
always answer.

| Model | Looks at | Stored as |
|---|---|---|
| 1-gram (unigram) | nothing | `model[word]` |
| 2-gram (bigram) | the last word | `model["w1"][w2]` |
| 3-gram (trigram) | the last 2 words | `model["w1 w2"][w3]` |
| … | … | … |
| 10-gram | the last 9 words | `model["w1 w2 … w9"][w10]` |

Everything else (the corpus, the cleaning rules, the tokenizer and
`weighted_pick`) is taken unchanged from the instructor notebooks for Sessions 1–3.

As in Session 3 there are two parts, and the only thing Part B takes from Part A
is the files in `models/`.

---
# Part A — The Data Engine

## A0. The tools from Sessions 1 and 2

Copied unchanged from the instructor's Session 3 notebook. The only difference
is `CORPUS_DIR`, which points at the `corpus/` folder next to this notebook.

Note that `list_corpus_files` keeps only `.txt` files, so, as in Session 3, the
corpus is the nine Gutenberg novels and `soda_dialogue_10000.csv` is not read.

In [16]:
import json
import os
import random

CORPUS_DIR = "corpus"
MODEL_DIR = "models"
MAX_N = 10                                  # the biggest model we build

START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"
APOSTROPHES = "'’‘´"        # ASCII ', curly ’ ‘, acute ´


def weighted_pick(weights):
    """Pick one key from {outcome: weight}, in proportion to its weight."""
    total = 0
    for outcome in weights:
        total += weights[outcome]

    target = random.random() * total

    running = 0
    for outcome in weights:
        running += weights[outcome]
        if target < running:
            return outcome
    return outcome


def list_corpus_files(folder):
    """Return a sorted list of the .txt file names in `folder`."""
    names = []
    for name in os.listdir(folder):
        if name.endswith(".txt"):
            names.append(name)
    names.sort()
    return names


def read_text_file(path):
    """Return the whole contents of a text file as one string."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def strip_boilerplate(text):
    """Return only the part of `text` between the Gutenberg marker lines."""
    lines = text.split("\n")

    body_lines = []
    inside = False
    found_start = False
    for line in lines:
        if line.startswith(START_MARKER):
            inside = True
            found_start = True
            continue
        if line.startswith(END_MARKER):
            inside = False
            continue
        if inside:
            body_lines.append(line)

    if not found_start:
        print("   WARNING: no start marker found, keeping the whole file")
        return text

    body = ""
    for line in body_lines:
        body += line + "\n"
    return body


def clean_word(word):
    """Normalise one raw token into a comparable word (possibly empty)."""
    result = ""
    for ch in word:
        if ch.isalpha():
            result += ch.lower()
        elif ch in APOSTROPHES:
            result += "'"
    return result.strip("'")


def tokenize(text):
    """Turn a block of text into a list of cleaned words, in order."""
    words = []
    for chunk in text.split():
        for piece in chunk.split("-"):
            word = clean_word(piece)
            if word != "":
                words.append(word)
    return words


def load_corpus(folder):
    """Read every book in `folder`, strip boilerplate, return one token list."""
    all_tokens = []
    for name in list_corpus_files(folder):
        raw = read_text_file(os.path.join(folder, name))
        tokens = tokenize(strip_boilerplate(raw))
        print("   %-14s %8d tokens" % (name, len(tokens)))
        all_tokens += tokens
    return all_tokens


def count_words(tokens):
    """Return {word: how many times it appears} -- the unigram model."""
    counts = {}
    for word in tokens:
        counts[word] = counts.get(word, 0) + 1
    return counts


def join_words(words):
    """Join a list of words into one single-spaced string."""
    out = ""
    for i in range(len(words)):
        if i > 0:
            out += " "
        out += words[i]
    return out


def save_model(counts, filename):
    """Write a counts dictionary to models/filename. Return its size in MB."""
    os.makedirs(MODEL_DIR, exist_ok=True)
    path = os.path.join(MODEL_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(counts, f)
    return os.path.getsize(path) / 1e6


def check(label, got, expected):
    """Compare your answer with the expected one and say which it is."""
    if got == expected:
        print("PASS ", label)
    else:
        print("FAIL ", label)
        print("      expected:", expected)
        print("      got:     ", got)


print("Session 1 and 2 tools loaded.")

Session 1 and 2 tools loaded.


---
## A1. One recipe for every n

In Session 3 each extra word of context meant one more level of nesting:

```python
bigrams["the"]["cat"]                 # 2 levels
trigrams["the"]["cat"]["sat"]         # 3 levels
```

Carrying on that way, the 10-gram would be **ten** levels deep, and we would
need a separate `add_…` function for every n. It also costs memory. Almost
every 9-word context in the corpus is unique, so a 10-deep nested dictionary
creates a chain of about eight tiny dictionaries for each of 1.5 million
contexts. On a laptop that runs out of memory.

So we keep the shape that `weighted_pick` needs, an **inner dictionary of
next words**, and change only the outer key. The context words are glued
together with spaces into one string:

```python
model["the"]            ->  {"cat": 2, "mat": 1}     # bigram: exactly as in Session 3
model["the cat"]        ->  {"sat": 1, "ran": 1}     # trigram
model["it was the"]     ->  {...}                    # 4-gram
```

That is two levels for every n. The bigram is **identical** to Session 3's,
and for higher n the only change is that the first few levels are flattened
into one key.

### Recipe: counting an n-gram

For every position `i` in the token list:

1. The **context** is the `n - 1` words `tokens[i : i + n - 1]`, joined with spaces.
2. The **next word** is `tokens[i + n - 1]`.
3. Count the next word into `counts[context]`, the same two steps as `add_bigram`.

`i` stops at `len(tokens) - n`, because the last `n - 1` words have nothing
`n - 1` places after them. (Check that this gives `len(tokens) - 1` for the
bigram and `len(tokens) - 2` for the trigram, as in Session 3.)

In [17]:
def context_key(words):
    """Turn a list of context words into the string used as a dictionary key."""
    return join_words(words)


def add_ngram(counts, context, next_word):
    """Record one sighting of `next_word` straight after the string `context`."""
    if context not in counts:
        counts[context] = {}
    inner = counts[context]
    inner[next_word] = inner.get(next_word, 0) + 1


def build_ngram_counts(tokens, n):
    """Return the n-gram model.

    n == 1  ->  {word: count}                     (the unigram, unchanged)
    n >= 2  ->  {"w1 ... w(n-1)": {wn: count}}
    """
    if n == 1:
        return count_words(tokens)
    counts = {}
    for i in range(len(tokens) - n + 1):
        context = context_key(tokens[i : i + n - 1])
        add_ngram(counts, context, tokens[i + n - 1])
    return counts

### Check it against Session 3

The same hand-traced sentence as Session 3. The bigram must come out
**exactly** as before. The trigram is Session 3's nested answer with its first
two levels joined into one key.

In [18]:
sentence = tokenize("the cat sat on the mat and the cat ran")

check("unigram = count_words", build_ngram_counts(sentence, 1), count_words(sentence))

check("bigram  = Session 3 bigram", build_ngram_counts(sentence, 2), {
    "the": {"cat": 2, "mat": 1},
    "cat": {"sat": 1, "ran": 1},
    "sat": {"on": 1},
    "on":  {"the": 1},
    "mat": {"and": 1},
    "and": {"the": 1},
})

check("trigram = Session 3 trigram, flattened", build_ngram_counts(sentence, 3), {
    "the cat": {"sat": 1, "ran": 1},
    "the mat": {"and": 1},
    "cat sat": {"on": 1},
    "sat on":  {"the": 1},
    "on the":  {"mat": 1},
    "mat and": {"the": 1},
    "and the": {"cat": 1},
})

check("4-gram", build_ngram_counts(sentence, 4), {
    "the cat sat": {"on": 1},
    "cat sat on":  {"the": 1},
    "sat on the":  {"mat": 1},
    "on the mat":  {"and": 1},
    "the mat and": {"the": 1},
    "mat and the": {"cat": 1},
    "and the cat": {"ran": 1},
})

# 10 tokens, so there is exactly one 10-gram and nothing longer.
check("10-gram", build_ngram_counts(sentence, 10),
      {"the cat sat on the mat and the cat": {"ran": 1}})
check("11-gram (too long)", build_ngram_counts(sentence, 11), {})

PASS  unigram = count_words
PASS  bigram  = Session 3 bigram
PASS  trigram = Session 3 trigram, flattened
PASS  4-gram
PASS  10-gram
PASS  11-gram (too long)


---
## A2. Build all ten models on the real corpus

### Estimate first

`unigram.json` was about half a megabyte, the bigram about 6 MB and the
trigram about 18 MB. Before you run the next cell, write down your guess for
the 10-gram.

We build one model at a time, save it, and throw it away (`del`) before the
next one, so only one big model is in memory at once. For each model we also
report how many contexts have **only one** possible next word. A die with one
face is not a die at all: the model can only copy what it read.

*This cell takes a minute or two and writes about 450 MB to `models/`.*

In [31]:
print("Reading the corpus...")
TOKENS = tokenize(read_text_file("corpus/soda_dialogue_10000.csv"))
print("total tokens:", len(TOKENS))
print()

print("  n   contexts   one-faced dice      file")
for n in range(1, MAX_N + 1):
    counts = build_ngram_counts(TOKENS, n)
    filename = "%dgram.json" % n
    size = save_model(counts, filename)

    if n == 1:
        print("%3d   %8d   %14s   %6.1f MB  models/%s"
              % (n, len(counts), "-", size, filename))
    else:
        one_faced = 0
        for context in counts:
            if len(counts[context]) == 1:
                one_faced += 1
        print("%3d   %8d   %13.1f%%   %6.1f MB  models/%s"
              % (n, len(counts), 100 * one_faced / len(counts), size, filename))
    del counts

del TOKENS

Reading the corpus...
total tokens: 1156182

  n   contexts   one-faced dice      file
  1      41450                -      0.7 MB  models/1gram.json
  2      41450            63.4%      3.7 MB  models/2gram.json
  3     244146            74.8%     11.1 MB  models/3gram.json
  4     567591            84.6%     22.0 MB  models/4gram.json
  5     820667            91.4%     33.0 MB  models/5gram.json
  6     969902            95.2%     42.6 MB  models/6gram.json
  7    1052035            97.3%     51.1 MB  models/7gram.json
  8    1096340            98.5%     58.6 MB  models/8gram.json
  9    1120755            99.1%     65.6 MB  models/9gram.json
 10    1134657            99.4%     72.2 MB  models/10gram.json


### Look at what you just made

- The number of contexts climbs fast and then levels off just under the number
  of tokens. By n = 10 almost **every** position in the corpus is its own,
  never-repeated context.
- The "one-faced dice" column heads towards 100%. A 10-gram that has seen
  `"it is a truth universally acknowledged that a single"` has seen it exactly
  once, followed by exactly one word. Rolling that die does not *generate*
  anything. It just **recites** the book.
- That is the trade-off from Session 3 in its most extreme form. A longer
  context gives more fluent text, gets stuck more often, makes bigger files and
  copies more of the source. Back-off is how we keep the fluency without
  getting stuck.

That is the end of the data engine. **Part B uses nothing from Part A except
the ten files in `models/`.**

---
# Part B — miniGPT with back-off

## B0. Load all the models

`MODELS[n]` is the n-gram model, so `MODELS[1]` is the unigram and
`MODELS[10]` is the 10-gram. All ten together need roughly 3–4 GB of memory.

In [32]:
def load_model(filename):
    """Load one counts file from the models/ folder."""
    path = os.path.join(MODEL_DIR, filename)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


MODELS = {}
for n in range(1, MAX_N + 1):
    MODELS[n] = load_model("%dgram.json" % n)
    print("loaded %2d-gram: %8d entries" % (n, len(MODELS[n])))

loaded  1-gram:    41450 entries
loaded  2-gram:    41450 entries
loaded  3-gram:   244146 entries
loaded  4-gram:   567591 entries
loaded  5-gram:   820667 entries
loaded  6-gram:   969902 entries
loaded  7-gram:  1052035 entries
loaded  8-gram:  1096340 entries
loaded  9-gram:  1120755 entries
loaded 10-gram:  1134657 entries


## B1. Back-off: when the big model is stuck, ask a smaller one

In Session 3 the trigram printed `[stuck after …]` and gave up. Now it asks
for help instead.

To choose the next word with a model of size `max_n`:

```
words so far: [..., "she", "said", "that", "it", "was", "not", "in", "the", "least"]

10-gram: is "she said that it was not in the least" a key of MODELS[10]?   no
 9-gram: is      "said that it was not in the least" a key of MODELS[9]?   no
 8-gram: is           "that it was not in the least" a key of MODELS[8]?   yes -> roll MODELS[8][...]
```

Each time the lookup fails we **drop the leftmost word** and try the next model
down, one word shorter.

### Recipe — `choose_next(models, words, max_n)`

1. `k = min(len(words), max_n - 1)` is the number of context words to start
   with. At the very start of a sentence we may have fewer than 9 words, and
   then we start lower.
2. While `k >= 1`:
   1. `context = context_key(words[len(words) - k :])`, the last `k` words.
   2. The model that looks at `k` words is `models[k + 1]`. If `context` is
      a key of it, roll that die and return `(the word, k + 1)`.
   3. Otherwise `k = k - 1` (drop the leftmost word) and try again.
3. If even the bigram has never seen the last word, fall back to the unigram:
   return `(weighted_pick(models[1]), 1)`. The unigram never gets stuck.

We return the order that answered as well as the word, so that we can see the
back-off happen.

In [4]:
def choose_next(models, words, max_n):
    """Choose the next word, backing off from the max_n-gram towards the unigram.

    Returns (next_word, n) where n is the order of the model that answered.
    """
    k = min(len(words), max_n - 1)
    while k >= 1:
        context = context_key(words[len(words) - k:])
        model = models[k + 1]
        if context in model:
            return weighted_pick(model[context]), k + 1
        k = k - 1                      # drop the leftmost word, ask a smaller model
    return weighted_pick(models[1]), 1


def generate_backoff(models, seed_words, length, max_n=MAX_N):
    """Generate `length` more words after `seed_words` using back-off.

    Returns (words, orders): orders[i] is the model order that chose the
    i-th *generated* word.
    """
    words = []
    for w in seed_words:
        words.append(w)
    orders = []
    for i in range(length):
        word, n = choose_next(models, words, max_n)
        words.append(word)
        orders.append(n)
    return words, orders

### Test it on a model small enough to trace by hand

Build models 1–4 from Session 3's sentence. Every context there has a
one-faced die, so the answers are certain and we can check them exactly.

In [9]:
toy = {}
for n in range(1, 5):
    toy[n] = build_ngram_counts(sentence, n)

# "sat on the" is a 4-gram context -> the 4-gram answers.
check("full context known",
      choose_next(toy, ["sat", "on", "the"], 4), ("mat", 4))

# "dog on the" is unknown to the 4-gram; "on the" is known to the trigram.
check("back off once",
      choose_next(toy, ["dog", "on", "the"], 4), ("mat", 3))

# "big red sat": 4-gram no, "red sat" trigram no, "sat" bigram yes.
check("back off twice",
      choose_next(toy, ["big", "red", "sat"], 4), ("on", 2))

# Only one word so far: start straight at the bigram.
check("short start",
      choose_next(toy, ["mat"], 4), ("and", 2))

# "ran" was never followed by anything -> nobody but the unigram can answer.
word, n = choose_next(toy, ["the", "cat", "ran"], 4)
check("all the way down to the unigram", n, 1)
check("unigram answer is a real word", word in toy[1], True)

# max_n limits how far up we start: with max_n = 2 it is just a bigram (plus unigram).
check("max_n = 2 behaves like a bigram",
      choose_next(toy, ["sat", "on", "the"], 2)[1], 2)

PASS  full context known
PASS  back off once
PASS  back off twice
PASS  short start
PASS  all the way down to the unigram
PASS  unigram answer is a real word
PASS  max_n = 2 behaves like a bigram


## B2. Generate with the 10-gram

Run it a few times and try your own opening words. The seed goes through the
same `tokenize` as the corpus, so `"It was"` becomes `["it", "was"]`.

In [33]:
def show(words, orders, seed_length):
    """Print the text, then the same text with each generated word tagged by its model order."""
    print(join_words(words))
    print()
    tagged = []
    for i in range(len(words)):
        if i < seed_length:
            tagged.append(words[i])
        else:
            tagged.append("%s[%d]" % (words[i], orders[i - seed_length]))
    print(join_words(tagged))


seed = tokenize("It was")
words, orders = generate_backoff(MODELS, seed, 40)
show(words, orders, len(seed))

it was nice getting to know you betterno problem it was nice talking to you too hi i'm looking for a car partwhat kind of part are you looking fori need a new enginewe don't have any engines in stock right now

it was nice[3] getting[4] to[5] know[6] you[7] betterno[8] problem[9] it[10] was[10] nice[10] talking[10] to[10] you[10] too[10] hi[10] i'm[10] looking[10] for[10] a[10] car[10] partwhat[10] kind[10] of[10] part[10] are[10] you[10] looking[10] fori[10] need[10] a[10] new[10] enginewe[10] don't[10] have[10] any[10] engines[10] in[10] stock[10] right[10] now[10]


### Which models actually did the work?

Generate a longer passage and tally the orders, using `tally`'s counting
pattern from Session 1.

In [34]:
words, orders = generate_backoff(MODELS, tokenize("the"), 500)

used = {}
for n in orders:
    used[n] = used.get(n, 0) + 1

print("order   words chosen")
for n in range(MAX_N, 0, -1):
    print("%4d    %5d  %s" % (n, used.get(n, 0), "#" * (used.get(n, 0) // 5)))

order   words chosen
  10      492  ##################################################################################################
   9        1  
   8        1  
   7        1  
   6        1  
   5        1  
   4        1  
   3        1  
   2        1  
   1        0  


Most words come from the 10-gram. Once the text has fallen into a passage of
some book, the 10-gram can follow it word for word. The lower orders are used
at the start, before there are 9 words of context, and whenever a one-faced
10-gram die leads somewhere the 10-gram has no die for.

## B3. Give it something it has never read

In Session 3 both models got stuck straight away on *motorbike*. With back-off
they cannot get stuck.

In [36]:
seed = tokenize("my motorbike")
words, orders = generate_backoff(MODELS, seed, 30)
show(words, orders, len(seed))

my motorbike for not doing itno sir it wouldn't be a good enough reason because i should have known better and made sure my homework was done before my dog got to

my motorbike for[1] not[2] doing[3] itno[4] sir[5] it[6] wouldn't[7] be[8] a[9] good[10] enough[10] reason[10] because[10] i[10] should[10] have[10] known[10] better[10] and[10] made[10] sure[10] my[10] homework[10] was[10] done[10] before[10] my[10] dog[10] got[10] to[10]


Read the tags. *motorbike* is in no model, so the first word has to come from
the unigram (`[1]`). After that the context fills with real words and the model
climbs back up through the orders.

## B4. Same models, different context lengths

`max_n` sets how many models we are allowed to use. `max_n = 2` is the Session 3
bigram (that never gets stuck), `max_n = 3` the trigram, and so on. Same seed,
same files. Only the amount of context changes.

In [37]:
seed = tokenize("she said that")
for max_n in [1, 2, 3, 4, 6, 10]:
    words, orders = generate_backoff(MODELS, seed, 25, max_n)
    print("max_n = %2d:  %s" % (max_n, join_words(words)))
    print()

max_n =  1:  she said that flirt and it do had to me besides me do seriously patrols or can i'm else might thank successful mean it's lot this sorry somethingexactly

max_n =  2:  she said that helpsyeah maybe i can't be more efficient but i wrote those things that it's hard to have a lot of my backpack some other day

max_n =  3:  she said that my page is enjoyable for me and i don't want to be friends but i'm fine i justi don't know where to startwell one thing

max_n =  4:  she said that my culture is very family oriented we believe that sustainable practices are not only good for the souli know i feel like i can't handle

max_n =  6:  she said that as long as our city remained hidden from the world we would never again know happiness or prosperity so we've been living in exile ever

max_n = 10:  she said that my dad is cheating on herthat's terrible what did she sayshe said that she found out that he has been meeting up with another woman



### To think about

- Somewhere between `max_n = 3` and `max_n = 10` the text stops being
  *English-like* and starts being *a quotation*. Where does it happen? Search
  the corpus files for one of the 10-gram's sentences and you will probably
  find it word for word. Session 1's ethics question (*whose writing is it?*)
  stops being abstract here.
- Our back-off is the simplest kind: the first model that knows the context
  wins outright, however little evidence it has. A 10-gram that saw a context
  **once** beats a 4-gram that saw it **five hundred** times. Real systems
  (Katz back-off, Kneser–Ney, "stupid back-off") weigh the evidence or mix the
  models together. How would you do it?
- The 10-gram file is about 100 MB and almost all of it is one-faced dice. Is
  it worth keeping? What would you lose if you deleted every context seen only
  once?
- Modern language models look at thousands of words of context and still do
  not "get stuck". They cannot be storing a table like ours. What must they be
  doing instead?

In [42]:
question = input("Prompt: ")
seed = tokenize("she said that")
words, orders = generate_backoff(MODELS, tokenize(question), 25, max_n)
print("max_n = %2d:  %s" % (max_n, join_words(words)))
print()

max_n = 10:  hi how are you todayi'm doing well thank you how are youi'm doing fine thank you did you need any help with anything todayno i don't think so but

